In [1]:
# 导入必要的库
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn import MSELoss
from plan_explorer import explore_candidate_plans_for_sql
from sklearn.model_selection import train_test_split
import json
from pprint import PrettyPrinter
from util.utils import load_data_from_db,evaluate_feature_discrimination,analyze_feature_importance
from Lero.model import LeroNet,LeroModel
from Lero.feature import AnalyzeJsonParser, FeatureGenerator
import numpy as np
import warnings
import os
import time
import scipy.stats as stats
import matplotlib.pyplot as plt
from model.Hpro_model import process_query_groups,ListwiseComparator
warnings.filterwarnings('ignore')

/home/windy/miniconda3/envs/optimizer/lib/python3.12/site-packages/torch/__init__.py:690: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


In [2]:
# 连接数据库并加载所有已探索计划的数据
db_config={
        "dbname":"tpcds10",
        "host":"localhost",
        "user":"windy",
        "password":"",
        "port":5432
    }
pp = PrettyPrinter()
query_groups = load_data_from_db(db_config)
print(f"每个查询组包含的字段:{query_groups[0].keys()}")
print(f"plans包含的字段:{query_groups[0]['plans'][0].keys()}")
pp.pprint(query_groups[:1])

每个查询组包含的字段:dict_keys(['query_name', 'plans', 'latencies'])
plans包含的字段:dict_keys(['Plan', 'Triggers', 'Planning Time', 'Execution Time'])
[{'latencies': [0.441, 0.459, 0.488, 1.867, 1.894, 120.0],
  'plans': [{'Execution Time': 1.646,
             'Plan': {'Actual Loops': 1,
                      'Actual Rows': 0,
                      'Actual Startup Time': 0.001,
                      'Actual Total Time': 0.001,
                      'Async Capable': False,
                      'Custom Plan Provider': 'DuckDBScan',
                      'DuckDB Execution Plan': {'blocked_thread_time': 0.0,
                                                'children': [{'children': [{'children': [{'children': [{'children': [{'children': [{'children': [{'children': [{'children': [],
                                                                                                                                                                'cpu_time': 0.0,
                                                

In [6]:
# 执行Lero二元计划比较模型的训练

    
# 全局配置（根据你的环境调整）
CUDA = torch.cuda.is_available()
device = torch.device("cuda" if CUDA else "cpu")
GPU_LIST = [0] if CUDA else []  # 按需调整GPU编号

# 关键：修复批次处理函数，过滤无效样本
def collate_pairwise_fn(batch):
    """处理批次数据，过滤空特征样本"""
    x1_list = []
    x2_list = []
    labels = []
    for x1, x2, label in batch:
        # 校验特征是否有效
        if hasattr(x1, 'get_feature') and hasattr(x2, 'get_feature'):
            if len(x1.get_feature()) > 0 and len(x2.get_feature()) > 0:
                x1_list.append(x1)
                x2_list.append(x2)
                labels.append(label)
    # 若当前批次无有效数据，返回空标识
    if not x1_list:
        return None, None, None
    return x1_list, x2_list, labels

# 主训练流程
lero_model_name = "lero_model.pth"
pre_training = False
ajp = AnalyzeJsonParser(normalizer=None, input_relations=[])
feature_generator = FeatureGenerator()
pairs = []
# plans_json = []

# 第一步：生成组内计划对（补充特征校验）
for group_idx, group in enumerate(query_groups):
    plans = group['plans']
    latencies = group['latencies']
    assert len(plans) == len(latencies), f"Group {group_idx}: Plans and latencies mismatch"
    num_plans = len(plans)

    # 组内两两生成计划对（i≠j）
    for i in range(num_plans):
        for j in range(num_plans):
            if i == j:
                continue
            try:
                # 提取并校验计划特征
                plan_i = plans[i]
                plan_j = plans[j]
                sam_i = ajp.extract_feature(plan_i)
                sam_j = ajp.extract_feature(plan_j)
                # 过滤空特征样本
                if len(sam_i.get_feature()) == 0 or len(sam_j.get_feature()) == 0:
                    print(f"Warning: Empty feature in group {group_idx}, pair ({i},{j})")
                    continue
            except Exception as e:
                print(f"Warning: Skip pair ({i},{j}) in group {group_idx}: {str(e)}")
                continue

            # 根据延迟生成标签
            lat_i = latencies[i]
            lat_j = latencies[j]
            label = 1.0 if lat_i >= lat_j else 0.0
            pairs.append((sam_i, sam_j, label))
            # plans_json.append((plan_i['plan_json'], plan_j['plan_json'], label))

if not pairs:
    raise ValueError("No valid training pairs generated from query groups")
print(f"Generated {len(pairs)} valid training pairs")

# 第二步：正确初始化模型
leronet = None
if not pre_training:
    # 从第一个有效样本获取特征维度
    input_feature_dim = len(pairs[0][0].get_feature())
    print("input_feature_dim:", input_feature_dim)
    # 用正确维度初始化模型
    leronet = LeroNet(input_feature_dim=input_feature_dim)

    # 多GPU配置（仅包装模型，不包装优化器）
    if CUDA:
        leronet = leronet.to(device)
        leronet = nn.DataParallel(leronet, device_ids=GPU_LIST)
else:
    # 预训练模式：需确保加载正确权重，这里补充示例逻辑
    raise NotImplementedError("Pre-training mode needs weight loading logic")

# 第三步：构建数据加载器
batch_size = 64
if CUDA and len(GPU_LIST) > 0:
    batch_size *= len(GPU_LIST)  # 多GPU合理放大batch size

dataset = DataLoader(
    pairs,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_pairwise_fn,
    drop_last=True  # 丢弃不完整批次，避免维度异常
)

# 第四步：初始化优化器（移除DataParallel包装）
optimizer = torch.optim.Adam(
    leronet.module.parameters() if (CUDA and isinstance(leronet, nn.DataParallel)) else leronet.parameters(),
    lr=1e-4  # 补充合理学习率，避免默认值可能的梯度爆炸
)

# 定义损失函数和激活函数
bce_loss_fn = torch.nn.BCELoss()
sigmoid = nn.Sigmoid()

# 第五步：训练循环（补充异常处理和设备适配）
losses = []
start_time = time.time()
leronet.train()

for epoch in range(100):
    loss_accum = 0.0
    valid_batches = 0

    for x1, x2, label in dataset:
        # 跳过空批次
        if x1 is None or x2 is None or label is None:
            continue
        valid_batches += 1

        # 构建树结构
        try:
            if CUDA and isinstance(leronet, nn.DataParallel):
                tree_x1 = leronet.module.build_trees(x1)
                tree_x2 = leronet.module.build_trees(x2)
            else:
                tree_x1 = leronet.build_trees(x1)
                tree_x2 = leronet.build_trees(x2)
        except Exception as e:
            print(f"Warning: Skip batch in epoch {epoch}: {str(e)}")
            continue

        # 前向传播
        # 模型为每个计划树输出一个浮点数评分(1维张量)
        y_pred_1 = leronet(tree_x1)
        y_pred_2 = leronet(tree_x2)
        # 使用张量差值标识这一次的预测标签,并映射到
        diff = y_pred_1 - y_pred_2
        prob_y = sigmoid(diff)

        # 处理标签：确保设备与模型一致，标签值只可能为0.0或1.0
        label_y = torch.tensor(
            np.array(label).reshape(-1, 1),
            dtype=torch.float64,
            device=device
        )

        # 计算损失
        loss = bce_loss_fn(prob_y, label_y)
        loss_accum += loss.item()

        # 反向传播与参数更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # 计算平均损失（避免除以0）
    if valid_batches > 0:
        avg_loss = loss_accum / valid_batches
        losses.append(avg_loss)
        print(f"Epoch {epoch}, training loss: {avg_loss:.6f}")
    else:
        print(f"Epoch {epoch}: No valid batches processed")

print(f"Training time: {time.time() - start_time:.2f}s, batch size: {batch_size}")

# 第六步：正确保存训练好的模型
lero_model = LeroModel(input_feature_dim=input_feature_dim, net=leronet)  
print("saving model...")
lero_model.save(lero_model_name)

Generated 4590 valid training pairs
input_feature_dim: 53


KeyboardInterrupt: 

In [3]:
# 执行Hpro列表式计划比较模型训练

# 1. 处理query_groups，生成训练组（每组20个计划）
plan_groups, latency_groups = process_query_groups(
    query_groups=query_groups,
    group_size=20,
    pad_latency=600.0
)

# 2. 初始化模型 暂定需要的中间状态维度为64
model = ListwiseComparator(hidden_size=4)
model.fit(plan_groups=plan_groups, latency_groups=latency_groups)

# 3. 训练模型
# model.fit(
#     plan_groups=plan_groups,
#     latency_groups=latency_groups,
#     epochs=100,           # 训练轮数
#     batch_size=16,       # 批次大小（根据GPU内存调整）
#     lr=1e-4,             # 学习率
#     weight_decay=1e-5,   # 权重衰减（L2正则）
#     log_interval=5       # 每5轮打印一次日志
# )

# # 5. 保存模型
# save_path = "./hpro_listwise_tpcds10.pth"
# model.save_model(save_path)
# print(f"模型已保存至：{save_path}")


处理完成：共生成 99 个训练组，每组 20 个计划
Epoch 1 step 10: loss=2.987659
Epoch 1 completed, avg loss=2.989878
Epoch 2 step 10: loss=2.987339
Epoch 2 completed, avg loss=2.988546
Epoch 3 step 10: loss=2.986425
Epoch 3 completed, avg loss=2.987569
Epoch 4 step 10: loss=2.984979
Epoch 4 completed, avg loss=2.986414
Epoch 5 step 10: loss=2.985499
Epoch 5 completed, avg loss=2.985238
Epoch 6 step 10: loss=2.986220
Epoch 6 completed, avg loss=2.983850
Epoch 7 step 10: loss=2.983661
Epoch 7 completed, avg loss=2.982449
Epoch 8 step 10: loss=2.979438
Epoch 8 completed, avg loss=2.980828
Epoch 9 step 10: loss=2.979145
Epoch 9 completed, avg loss=2.979306
Epoch 10 step 10: loss=2.976809
Epoch 10 completed, avg loss=2.977413
Epoch 11 step 10: loss=2.973031
Epoch 11 completed, avg loss=2.975043
Epoch 12 step 10: loss=2.971115
Epoch 12 completed, avg loss=2.973539
Epoch 13 step 10: loss=2.966449
Epoch 13 completed, avg loss=2.970540
Epoch 14 step 10: loss=2.967556
Epoch 14 completed, avg loss=2.967801
Epoch 15 st

2.303834089866051

In [ ]:
# 在测试集上,评估Postgres的遗传优化器,duckdb优化器,Lero学习排序优化器,Hpro优化器四者的端到端时延
# 测试集的查询语句的配置:
FILE_PATH = os.popen('pwd').read().strip()
print("当前文件路径:", FILE_PATH)
TEST_QUERIES_PATH = FILE_PATH + "/test_queries/"
test_config = {
    "tpc-ds": f"{TEST_QUERIES_PATH}/tpcds_queries.txt",
}
lero_model = LeroModel(None)
lero_model.load("lero_model.pth")

for workload, query_file in test_config.items():
    with open(query_file, 'r') as f:
        rows = f.readlines()
        row_num = len(rows)
        print(f"正在使用测试文件{query_file}测试{workload}的{row_num}个查询语句...")
        i = 0
        for q in rows:
            i += 1
            print(f"{workload}:{i}/{row_num}...")
            # plan , latency = explore_candidate_plans_for_sql(q)
            pass


In [ ]:
# 计划特征化阶段评估实验
# 比较项目:特征重要性、特征区分度
# hpro特征维度:128
# Lero特征维度:64

embedding_hpro = []
embedding_lero = []
lats = []
names = []
for group in query_groups:
    embedding_hpro += group['embeddings']
    embedding_lero += group['embeddings_lero']
    lats += group['latencies']
    names += group['query_name'] * len(group['embeddings'])

# 特征重要性评估
hpro = analyze_feature_importance(embeddings=embedding_hpro,latencies=lats)
lero = analyze_feature_importance(embeddings=embedding_lero,latencies=lats)
pp.pprint(f"Hpro模型特征重要性: {hpro['feature_importance']}")
pp.pprint(f"Lero模型特征重要性: {lero['feature_importance']}")



# diff_hpro = evaluate_feature_discrimination(embeddings=embedding_hpro, query_names=names)
# diff_lero = evaluate_feature_discrimination(embeddings=embedding_lero, query_names=names)
# pp.pprint(f"hpro特征区分度: {diff_hpro}")
# pp.pprint(f"Lero特征区分度: {diff_lero}")   


In [ ]:
# 对行列混合计划补充Lero模型的特征化编码结果

sample_entities = []
weight_path = "lero_feature_extractor_weights.pth"

ajp = AnalyzeJsonParser(normalizer=None,input_relations=[])
sam = ajp.extract_feature(query_groups[0]['plans'][0]['plan_json'])
input_feature_dim = len(sam.get_feature())
leronet = LeroNet(input_feature_dim=input_feature_dim)

if os.path.exists(weight_path):
    # 已经预训练过了
    leronet.load_state_dict(torch.load(weight_path))
    leronet.eval()
else:
    # 需要预训练Lero模型
    # 准备数据：将原始特征和延迟转换为张量
    # 假设sample_entities已收集所有查询计划的原始特征
    all_entities = []
    lats = []
    for group in query_groups:
        for i, plan in enumerate(group['plans']):
            sam = ajp.extract_feature(plan['plan_json'])
            all_entities.append(sam)
        lats += group['latencies']

    # all_entities = torch.tensor(all_entities, dtype=torch.float32)
    all_latencies = torch.tensor(lats, dtype=torch.float64).view(-1, 1)  # 目标变量（延迟）

    # 划分训练集和验证集
    train_entities, val_entities, train_lats, val_lats = train_test_split(
        all_entities, all_latencies, test_size=0.2, random_state=42
    )

    # 定义优化器和损失函数
    optimizer = optim.Adam(leronet.parameters(), lr=1e-4)
    criterion = MSELoss()  # 回归任务：预测延迟

    num_epochs = 30
    leronet.train()  # 切换到训练模式

    # 训练阶段
    for epoch in range(num_epochs):
        
        optimizer.zero_grad()  # 清空梯度
        # 前向传播：输入原始特征，输出嵌入（可直接用于预测延迟，或通过一个小头部预测）
        # 假设LeroNet的输出嵌入通过一个线性层预测延迟
        train_trees = leronet.build_trees(train_entities)  # 构建树结构（与特征提取阶段一致）
        embeddings = leronet.extract_features(train_trees)  # 传入树结构，而非原始特征
        preds = torch.nn.Sequential(
            nn.Linear(64, 32),
            nn.LeakyReLU(),
            nn.Linear(32, 1)
        )(embeddings)  # 简单头部：从嵌入预测延迟
        loss = criterion(preds, train_lats)  # 计算预测损失
        
        # 反向传播与参数更新
        loss.backward()
        optimizer.step()
        
        # 验证阶段（每轮epoch结束后）
        leronet.eval()  # 切换到评估模式
        with torch.no_grad():
            val_trees = leronet.build_trees(val_entities)
            val_embeddings = leronet.extract_features(val_trees)
            val_preds = torch.nn.Linear(64, 1)(val_embeddings)
            val_loss = criterion(val_preds, val_lats)
        
        # 打印日志
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")
        leronet.train()  # 切回训练模式

leronet.eval()  # 切换到评估模式

for group in query_groups:
    sample_entities = []
    for i, plan in enumerate(group['plans']):
        sam = ajp.extract_feature(plan['plan_json'])
        sample_entities.append(sam)
        
    trees = leronet.build_trees(sample_entities)
    features = leronet.extract_features(trees)
    # 此时的trees是tensor([[...],[64;float],...])
    features = features.detach().numpy().tolist()
    assert len(features) == len(group['embeddings'])
    group['embeddings_lero'] = features
    # print(f"查询组{group['query_name']}补充Lero特征化编码完成:{group['embeddings_lero'][:2]}")

torch.save(leronet.state_dict(), weight_path)
print(f"Lero模型权重已保存到{weight_path}")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示异常问题
hpro_importance = np.array(hpro['feature_importance'])

lero_importance =  np.array(lero['feature_importance'])
# 子图1：HPRO特征重要性分布（直方图+KDE）
plt.subplot(1, 2, 1)
# 直方图（透明度0.5，展示原始分布）
n_bins = 50
plt.hist(hpro_importance, bins=n_bins, alpha=0.5, color='#2E86AB', label='histogram')
# KDE曲线（平滑概率密度，突出分布趋势）
kde_hpro = gaussian_kde(hpro_importance)
x_hpro = np.linspace(hpro_importance.min(), hpro_importance.max(), 200)
plt.plot(x_hpro, kde_hpro(x_hpro), color='#A23B72', linewidth=2.0, label='KDE')
# 标注关键信息
max_hpro = hpro_importance.max()
plt.axvline(x=max_hpro, color='#F18F01', linestyle='--', linewidth=2, label=f'Max importance: {max_hpro:.3f}')
plt.title('HPRO Distribution (128-d)', fontsize=14, fontweight='bold')
plt.xlabel('Feature importance', fontsize=12)
plt.ylabel('Probability Density/Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(alpha=0.3)

# 子图2：Lero特征重要性分布（直方图+KDE）
plt.subplot(1, 2, 2)
# 直方图
plt.hist(lero_importance, bins=n_bins, alpha=0.5, color='#6A994E', label='histogram')
# KDE曲线
kde_lero = gaussian_kde(lero_importance)
x_lero = np.linspace(lero_importance.min(), lero_importance.max(), 200)
plt.plot(x_lero, kde_lero(x_lero), color='#BC4749', linewidth=2.0, label='KDE')
# 标注关键信息
max_lero = lero_importance.max()
plt.axvline(x=max_lero, color='#F18F01', linestyle='--', linewidth=2, label=f'Max importance: {max_lero:.3f}')
plt.title('Lero Distribution (64-d)', fontsize=14, fontweight='bold')
plt.xlabel('Feature importance', fontsize=12)
plt.ylabel('Probability Density/Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(alpha=0.3)

# 整体布局调整
plt.tight_layout()